<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

# 25 — Scene 1 / not-Scene 1 sorter (widget-free)

Sort the contact sheets from `scripts/scene_sampler.py` into **Scene 1** (front-on Mushroom view) or **Not Scene 1**. Each sheet is shown as a **static image** and filed with a short function call.

**This version uses no ipywidgets**, so it is immune to the JupyterLab real-time-collaboration issue (`jupyter_server_documents`) that stopped the button widget from rendering. Static `imshow` output is unaffected.

**How to use**

1. Set `SESSION_DIR` in the config cell to the session you want to sort.
2. Run the config cell and the engine cell once (top to bottom). The first sheet appears.
3. In the **driver cell** at the bottom, call one of these, then press **Shift+Enter**:
   - `s1()` — file as **Scene 1**
   - `no()` — file as **Not Scene 1**
   - `sk()` — **Skip** (leave in place, revisit next session)
   Each call files the current sheet, logs it, and draws the next one. Just edit the call and Shift+Enter again for each sheet.
4. Helpers: `show()` redraws the current sheet without filing; `undo()` moves the most recently filed sheet back for re-review (after a misclick).

Every decision is appended to `sort_log.csv` (an audit trail). Sorting is **resumable** — filed sheets leave `contact_sheets/`, so re-running picks up where you left off. Final pile counts come from the folder contents (`sorter.counts`), not by replaying the log.

The move/log logic lives in `scripts/scene_sorter.py` (unit-tested); this notebook is only the UI.

</span>

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import matplotlib.pyplot as plt
import scene_sorter as sorter

# --- CONFIG: point this at the session you want to sort ---------------------
SESSION_DIR = Path.cwd().parent / "scene_sorting" / "validation_2023_03"

print("Sorting:", SESSION_DIR)
print("Status :", sorter.counts(SESSION_DIR))

In [ ]:
# --- Widget-free sorting engine --------------------------------------------
import csv

_skipped: set[str] = set()  # sheet filenames skipped this session


def _pending():
    return [p for p in sorter.pending_sheets(SESSION_DIR) if p.name not in _skipped]


def show():
    """Redraw the current (next-to-sort) contact sheet, full size."""
    plt.close("all")
    rem = _pending()
    c = sorter.counts(SESSION_DIR)
    print(
        f"Scene 1: {c['scene1']}   |   Not Scene 1: {c['not_scene1']}   |   "
        f"remaining: {len(rem)}   (skipped this session: {len(_skipped)})"
    )
    if not rem:
        print("\u2705 Nothing left to sort in this session.")
        return
    sheet = rem[0]
    _fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(plt.imread(sheet))
    ax.axis("off")
    ax.set_title(sheet.stem, fontsize=11)
    plt.show()


def _decide(decision):
    rem = _pending()
    if not rem:
        show()
        return
    sheet = rem[0]
    if decision == "skip":
        _skipped.add(sheet.name)
    sorter.apply_decision(SESSION_DIR, sheet, decision)
    show()


def s1():
    """File the current sheet as Scene 1, then show the next."""
    _decide("scene1")


def no():
    """File the current sheet as Not Scene 1, then show the next."""
    _decide("not_scene1")


def sk():
    """Skip the current sheet (leave it in place), then show the next."""
    _decide("skip")


def undo():
    """Move the most recently filed sheet back to contact_sheets/ for re-review."""
    paths = sorter.session_paths(SESSION_DIR)
    if not paths.log_csv.exists():
        print("No decisions logged yet \u2014 nothing to undo.")
        return
    rows = list(csv.DictReader(paths.log_csv.open()))
    filed = [r for r in rows if r["decision"] in sorter.FILED_DECISIONS]
    if not filed:
        print("Nothing filed yet \u2014 nothing to undo.")
        return
    last = filed[-1]
    name = last["stem"] + ".png"
    src = paths.target_for(last["decision"]) / name
    if not src.exists():
        print(f"Cannot undo \u2014 {name} is not in {last['decision']}/ (already undone?).")
        return
    src.rename(paths.contact_sheets / name)
    _skipped.discard(name)
    print(f"Moved back: {name}  (was filed as {last['decision']})")
    show()


print(
    "Engine ready.  s1() = Scene 1   no() = Not Scene 1   sk() = Skip\n"
    "Edit the call in the driver cell below and press Shift+Enter for each sheet."
)
show()

In [ ]:
# DRIVER CELL — look at the sheet above, set your call, press Shift+Enter.
#   s1() = Scene 1   |   no() = Not Scene 1   |   sk() = Skip   |   undo() = fix last misclick
no()